In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-12: The teacher, and when it is not enough

## RFT Alternative methods, part 2 of the RLLM series


## Where this session fits

> **RLLM-11 gave the mechanism: the student samples, a teacher reads the student's prefix, a per-position loss. It assumed a good teacher. This session: where the teacher's strength comes from, and when it is not enough.**

- **Part 2: RFT Alternative methods**: alternatives to the PPO and GRPO you know, each placed by three axes
- **Axes**: 1 whose samples, 2 who supplies the weights, 3 how fine. No new method: we test the teacher of OPD (on-policy distillation, RLLM-11)
- **Position**: session 12 of 14, after RLLM-11 (On-policy distillation: the mechanism)
- **Goal of Part 2**: a **map** that places any method by the three axes

<img src="images/series_map_12.png" width=55%>


## This session: RLLM-12

- **Question**: where does a teacher's strength come from, and **when is it not enough**?
- **By the end you can**:
  - name **four sources** of a teacher's strength and what each can cost, and say what **privileged information** is
  - state the **two tests** of a teacher: the **quality floor** (beat the student by more than the noise) and the **struggle set** (failed prompts)
  - state the **open premise** (at a prefix with the student's mistake, does the teacher repair or continue it?) and what would settle it
  - name what OPD **may not reach**, and use a table to decide when to reach for OPD


## A stronger teacher does not help by itself

> **A teacher helps only if it clears two tests: it beats the student by more than the noise, and it is informative where the student struggles.**

<img src="images/teacher_floor_and_struggle.png" width=60% alt="left: a student at 60% with a noise band, teacher X inside the band and teacher Y above it; right: teacher A has the higher average but 20% on the student's failures, teacher B 65%">

- **Noise**: how far an accuracy moves **without any real difference**, in two parts (RLLM-14's names):
  - **run-to-run noise**: retrain with another seed, or **re-evaluate** (score the same model again)
  - **prompt-set noise**: only $n$ prompts. One accuracy $\hat{p}$ has standard error $\sqrt{\hat{p}(1-\hat{p})/n}$, about **3.5 points** at $n=200$, 60%
- Teacher X's 2-point lead is **inside** that noise
- **Struggle set**: the prompts the **student** gets wrong. Teacher A's higher average hides 20% there
- Numbers are **invented**. First, where a teacher's strength comes from


## Step 1 of 5: A teacher can be stronger in four ways

**Why this section?** The two tests ask whether a teacher is strong enough. Where its strength comes from decides what it can give and what it can cost.

<img src="images/teacher_sources.png" width=70% alt="a grid: the student row has nothing different; each source (1 to 4) changes size, training or prompt context">

1. **More parameters**: a bigger model
2. **More training** (including RL)
3. **Extra context**: examples or hints in the teacher's prompt
4. **A later checkpoint** of the student's own training run

- Sources 1, 2 and 4 change the **parameters**; source 3 only the **input**
- A real teacher often differs in **several** ways at once


## Step 1 of 5: What each source buys, and what it can cost

| Source | What it can give | A question to ask |
|---|---|---|
| 1 More parameters | capacity the student lacks | same **tokenizer and format**? Affordable? |
| 2 More training (incl. RL) | skill on the **trained** task | trained on what? A **proxy** reward (RLLM-1) leaves its flaws in the teacher |
| 3 Extra context | the procedure or answer, with no new parameters | is the context **correct**? Where does it come from? The quality floor (Test 1) is **vacuous** here |
| 4 Later checkpoint | the student's own future | it may **share the student's mistakes** |

- **Ground-truth dependence**: extra context often **is** the true answer. State it openly
- **Source 3 and the floor**: without the context the teacher equals the student. With a reference solution it scores (illustrative) about 100% by copying
  - compare instead its per-token distribution on the student's **own** reasoning, **with and without** the context (RLLM-14's probe)


## Step 1 of 5: Extra context is privileged information

- **Privileged information**: hints the **student lacks at test time**, which the **teacher** is given while the student trains
  - a worked example, a hint, a rubric, a reference solution, or tool output
- The teacher reads the prompt $\x'$ (context **plus** task); the student reads only $\x$
- The **strongest** variant, a reference solution, gives the teacher the **true answer**, and with it the most ground-truth dependence
- **Main risk, the privileged-information gap** (our reading, of a well-known gap): the teacher's distribution depends on information the student can **never have at test time**
  - the student is trained to match behavior it **cannot reproduce**. It may learn confident jumps to the answer
- A hint may also be **wrong**, and the student then inherits the error
- **Warning on words**: papers call one model taught with extra context **self-distillation**. RLLM-9's is the keep-what-passes filter loop


## Step 2 of 5: Test 1, the quality floor: beat the student by more than the noise

- **No universal threshold.** The **quality floor**: the teacher's accuracy must exceed the student's by **more than the noise**
  - same **scorer** (reward function or verifier), same **prompts**
  - a gap of two **independent** accuracies has standard error about $3.5\sqrt2\approx4.9$ points
  - on the **same** prompts, a **paired bootstrap** (resample prompts, recompute the gap) gives its spread
- Below the floor a gain is **unlikely** (our inference). The student may take on the teacher's flaws
- A **format floor**: a teacher that cannot produce your format gives a useless reference
- **Not for source 3**: the floor is **vacuous** there. RLLM-14's planted-error probe is the check

> **Keep this: the quality floor. A teacher helps only if it beats the student by more than the noise. It returns in RLLM-14 as the test of any comparison.**
<!-- seed-plant: quality-floor -->


## Step 2 of 5: Test 2, the struggle set: good where the student needs help

- The loss needs a teacher that is **informative at the student's prefixes**: its next-token distribution points toward **better completions** there
- A stand-in we can measure: the teacher's accuracy on the **struggle set**, against the student's **re-evaluated** accuracy on that same set
  - the set is chosen on failures, so the student's first score there is too low (regression to the mean, RLLM-14)
  - "good on average" and "informative here" are **two claims**. The loss cannot tell you which holds
- The stand-in judges **answers**, not the teacher at the student's **wrong prefix**. That gap is the **open premise**, step 3
- A teacher can pass both tests and still disappoint at a prefix with a mistake


## Step 3 of 5: An open premise: what does a teacher say at a prefix with a mistake?
<!-- seed-pay: unreached-state -->

**Why this section?** The loss reads the teacher at the student's prefixes, and some of them hold the student's mistakes. Is the teacher good there?

> **A hypothesis: no data in this series supports either reading yet.**

<img src="images/teacher_at_mistake.png" width=52% alt="the student's prefix contains a wrong step; reading 1 the teacher repairs, reading 2 it continues the mistake">

- From RLLM-11: the teacher speaks at states the **student** reaches, including prefixes with a mistake, a state the teacher itself would likely never reach
- **Reading 1, repair**: at later positions its distribution points at the correct step, so the student learns to recover
- **Reading 2, continue**: it points at a step consistent with the mistake, so the student is pulled along the wrong chain
  - **our reading** of why it might: models are trained to stay consistent with what is written. A plausible mechanism, not a measured result


## Step 3 of 5: What the loss does at the mistake, and what would settle it

- Where the student **writes** the mistake the teacher reads a clean prefix, so the loss lowers the wrong token there
  - each position compares distributions **at that position only**
- A whole-completion signal (a reward in GRPO, a good-versus-bad pair in DPO) can mark the completion that **contains** the mistake, not where. **Our reading**
- **What would settle it**: planted-error prefixes, per teacher, task and context (RLLM-14)
- **Predictions**: write down yours now. Does the teacher repair or continue? Does a reference solution change it?


## Step 4 of 5: What OPD may not reach

**Why this section?** Even a teacher that passes both tests leaves limits, set by how the loss is built.

<img src="images/opd_reach.png" width=44% alt="the student visits some regions of prefixes rarely, so little teacher signal there">

- **Across a population**: signal only at prefixes the student visits
  - SFT fixes its prefixes in advance; with OPD the student's sampling decides
- **Within one completion, if reading 2 holds**: after a wrong step, later positions' signals go along with the mistake (step 3)
- **Above the teacher**: the loss pulls the student **toward** the teacher, not past it
  - exceeding it needs a signal from outside, a reward say. Whatever the teacher lacks, the student inherits


## Step 4 of 5: Adding a reward to OPD: a hybrid, and a mixed loss

> **A reward-filtered hybrid takes its weights from the teacher: its loss has **no term that pays for exceeding it** (our reading, untested). A mixed loss does: its RL term can reward outcomes the teacher never reaches.**

- **Hybrid** (RLLM-9's one meaning): a **reward picks** the data and a **reference trains** on it
  - for OPD: keep only the student's completions that pass a checker, then apply the OPD loss to them
  - on the map: the OPD box, with a 0 or 1 weight chosen by a reward
- **Mixed loss** (not a hybrid): $\loss=\loss_{\text{RL}}+\lambda\,\loss_{\text{OPD}}$
  - $\loss_{\text{RL}}$ is the reward-driven loss (GRPO's, say); $\lambda$ is a **mixing coefficient**
  - the teacher shapes **every position**; the reward judges the **whole completion**
  - on the map: a mix of two cells, the reward cell and the OPD cell, not a cell of its own


## Step 5 of 5: When would you reach for OPD?

**Why this section?** The tests and limits of steps 2 to 4 turn into a table about OPD alone, argued from **how it is built**. The full decision tool, across all methods, is RLLM-13.

| Situation | Lean | Why |
|---|---|---|
| Competent teacher (both tests passed; the premise is open), no good scorer | **OPD** | its one dependency, a good teacher, is met |
| Many acceptable answers, to keep | **forward KL** | it must cover the spread (RLLM-11) |
| Early mistakes the teacher cannot undo | a **whole-completion signal** (GRPO's reward, DPO's pairs) | OPD reads each position alone; real only if reading 2 holds |
| Teacher inside the noise, or poor on the struggle set | not OPD yet | a gain is unlikely (our inference), and the student may take on its flaws; equal accuracy with complementary strengths or a format to transfer can still help |

Conditionals, not a ranking.


## Think about

1. **Predict first**: a teacher scores 82% on average and the student 60%. Which **further number** would you ask for before trusting it?
2. For each of the **four sources**, name **one reason its strength might not transfer** to **your** student
3. A **hint** given to the teacher is wrong on 10% of prompts. What can the student learn on those?
4. **Predict**: given a reference solution, does the teacher repair or continue a planted error? Say why. RLLM-14 asks how to **check** it


## What we established, and what it sets up

- **Established**: a teacher needs the **quality floor** and **struggle set**. It has **four sources**, one **privileged information**. Three **limits** bound OPD
- **Limit of this session**: OPD's reach rests on an **open** premise: if a teacher **continues** a prefix's mistake, later positions' signals follow it
- **Sets up**: RLLM-13, The map and the decision tool
  - RLLM-13: the map and the decision tool. RLLM-14: the checks (**how would I check any of this?**)


## Reading for this session

- **Suggested**: for a paper, at least the **abstract and introduction**. For other items, the parts named
  - [Zhao et al. (2026), *Self-Distilled Reasoner: On-Policy Self-Distillation for Large Language Models*](https://arxiv.org/abs/2601.18734): abstract and introduction: a model as its own teacher
  - [Hübotter et al. (2026), *Reinforcement Learning via Self-Distillation*](https://arxiv.org/abs/2601.20802): abstract and introduction: the model, shown feedback, is its own teacher
- **Notes**
  - These papers use *self-distillation* for one model as its own teacher, with extra context.
  - Here that is OPD with an extra-context teacher, not RLLM-9's loop.
  - *RLSD* is our name for RL alternating with RLLM-9's loop; neither paper is RLSD.
  - Read the Additional items after you write down your predictions.
- **Additional** (3 more, for digging deeper), and the other sources this session cites: `reading_list.ipynb`
- Links were checked in October 2026; tell us if one has moved


### Aside, justification: the variants of extra context

*For your own review; we will not work through it in session.*

The teacher reads the prompt $\x'$ (context **plus** task); the student reads only $\x$.

| Variant | What the teacher gets | What it can give |
|---|---|---|
| **In-context examples** | solved pairs | format and method, not this answer |
| **Hints** | a nudge for **this** prompt | a direction; the hint may be wrong |
| **Instructions or rubric** | the procedure | the procedure, if followed |
| **Reference solution** | the **true answer** | the most, and the most ground-truth dependence |
| **Tool output or retrieval** | outside facts | facts the parameters lack |

- Say which variant you use, and where its context **comes from**


### Aside, skipped: RLSD, RL alternating with a self-distillation phase

*For your own review; we will not work through it in session.*

- **RLSD** is **our name for alternating RL with a self-distillation phase; not a published method name; neither paper calls it RLSD**
- $K$ GRPO steps, then a round of **sample, filter, imitate** (RLLM-9), repeated
  - **two loops in sequence**, not a new mechanism; **no teacher** here
  - on the map (RLLM-13): not a box of its own, but a **move over time** between the reward column (GRPO) and the hybrid box (RLLM-9)
  - the papers use *self-distillation* for a model taught with extra context, a different thing (see the reading note)
- Open: does the **negative** weight during the RL phase **protect** the method, or does the next imitation round **erase** what it bought?
